### Mask-Based Implementation
The most direct implementation follows the formula exactly: compute all attention scores, apply the mask, run softmax, and aggregate values.

In [ ]:
import numpy as np

def softmax(x,axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def create_sliding_window_mask(seq_len, window_size, causal=False):
    """
    Create a sliding window attention mask.

    Args:
        seq_len: Length of the sequence
        window_size: Size of the attention window
        causal: If True, mask future positions (for autoregressive models)

    Returns:
        Mask of shape (seq_len, seq_len) with 0 for allowed positions,
        -inf for masked positions
    """

    mask = np.full((seq_len, seq_len), -1e9)
    half_window = window_size//2

    for i in range(seq_len):
        if causal:
            # Only attend to current and previous positions within window
            start = max(0, i - window_size + 1)
            end = i + 1
        else:
            start = max(0, i - half_window)
            end = min(seq_len, i + half_window + 1)

                
        mask[i, start:end] = 0

    return mask     

Why is $-1e9$ used?In Self-Attention mechanisms (such as Causal / Look-ahead Attention or padding masks), attention scores are calculated using the Softmax function:$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}$$When you add $-1e9$ to an attention score before applying Softmax:$$e^{-1e9} \approx 0$$Setting a masked position's score to $-1e9$ causes its output probability to become 0, completely blocking the model from paying attention to that token (e.g., future words during auto-regressive generation, or padding tokens).

`if causal:
            # Only attend to current and previous positions within window
            start = max(0, i - window_size + 1)
            end = i + 1`

This snippet calculates the slice bounds [start, end) of tokens that position $i$ is allowed to attend to in Sliding Window Causal Attention (also known as Local Causal Attention, used in models like Mistral 7B and Longformer).

Variable Breakdown
- end = i + 1: Enforces the causal constraint. Token $i$ can only attend to past and current tokens up to index $i$ (since Python upper slice bounds are exclusive). Future tokens ($> i$) are excluded.
- start = max(0, i - window_size + 1): Enforces the window size constraint. Token $i$ is restricted to looking back at most window_size tokens (including itself). The max(0, ...) guard prevents negative indexing at the start of the sequence.

Step-by-Step Example

With window_size = 3:


| Current Position ($i$) | `start` | `end` | Attended Token Indices | Window Span |
| --- | --- | --- | --- | --- |
| **0** | `max(0, 0-3+1) = 0` | `1` | `[0]` | 1 token (growing window) |
| **1** | `max(0, 1-3+1) = 0` | `2` | `[0, 1]` | 2 tokens (growing window) |
| **2** | `max(0, 2-3+1) = 0` | `3` | `[0, 1, 2]` | 3 tokens (full window reached) |
| **5** | `max(0, 5-3+1) = 3` | `6` | `[3, 4, 5]` | 3 tokens (sliding window) |

`else:
    # Symmetric window
    start = max(0, i - half_window)
    end = min(seq_len, i + half_window + 1)`

This snippet calculates the index bounds [start, end) for Non-Causal (Symmetric / Bidirectional) Sliding Window Attention, commonly used in encoder models (like Longformer or BigBird) and vision transformers.

Unlike the causal variant, token $i$ can look both backward into the past and forward into the future within a fixed neighborhood radius (half_window).).

Variable Breakdown

- **start = max(0, i - half_window)**: Computes the left boundary. Looks back up to `half_window` positions before index $i$. `max(0, ...)` ensures indices don't drop below 0 at the beginning of the sequence.
- **end = min(seq_len, i + half_window + 1)**: Computes the right boundary. Looks ahead up to `half_window` positions after index $i$ (`+1` accommodates Python's exclusive slice upper bound). `min(seq_len, ...)` prevents indexing beyond the end of the sequence.

Total maximum window size = $2 \times \text{half\_window} + 1$ tokens (left context + current token + right context).

Step-by-Step Example

With `seq_len = 10` and `half_window = 2` (total context window size = $2 \times 2 + 1 = 5$ tokens):

| Position ($i$) | `start` | `end` | Attended Indices | Behavior |
| --- | --- | --- | --- | --- |
| **0** (Start) | `max(0, 0-2) = 0` | `min(10, 0+2+1) = 3` | `[0, 1, 2]` | Truncated left edge |
| **1** | `max(0, 1-2) = 0` | `min(10, 1+2+1) = 4` | `[0, 1, 2, 3]` | Partial left edge |
| **5** (Middle) | `max(0, 5-2) = 3` | `min(10, 5+2+1) = 8` | `[3, 4, 5, 6, 7]` | Full symmetric window |
| **9** (End) | `max(0, 9-2) = 7` | `min(10, 9+2+1) = 10` | `[7, 8, 9]` | Truncated right edge |


### Compute sliding window attention.

In [ ]:
def sliding_window_attention(Q, K, V, window_size, causal=False):
    """
    Compute sliding window attention.

    Args:
        Q: Query matrix of shape (seq_len, d_k)
        K: Key matrix of shape (seq_len, d_k)
        V: Value matrix of shape (seq_len, d_v)
        window_size: Size of the attention window
        causal: If True, use causal masking

    Returns:
        Output of shape (seq_len, d_v)
    """
    seq_len, d_k = Q.shape

    # Compute attention scores
    scores = Q @ K.T / np.sqrt(d_k)

    # Apply sliding window mask
    mask = create_sliding_window_mask(seq_len, window_size, causal)
    masked_scores = scores + mask

    # Apply softmax
    weights = softmax(masked_scores, axis=-1)

    # Compute weighted sum of values
    output = weights @ V

    return output, weights

This function follows our four-step process exactly: compute scores (Q@K.T), scale by dk, add the mask, apply softmax, and multiply by values. The function returns both the output and the attention weights, which we'll visualize to verify the windowed pattern.